# 演示完案例，再讲使用规范
1. 不能用 try/catch包裹， 因为intrrupt就是用抛出异常去暂停，抛不出去，就不能计算了，正确是其他地方try catch
2. 不能用if条件判断，影响interrupt的逻辑，   让 12 3， 13 的顺序有问题， resmue会错误语义混乱， 应该拆到不同的节点
    - 要么拆到多个不同的节点
3. 不能用不确定的for循环， 添加interrupt 也不行， 可能断电数量混乱，语义对不上，因为检查点回溯来完成功能，
4. interrupt的入参不能传入不能序列化的复杂内容，  不能JSON序列化， 正确的是 传递一些简单的字符串， 数字 ，数字接收， 或者仅包含字典的内容，  不能填函数不能填类
5. 断点之前的副作用是幂等的，   就是想要恢复中断执行，前面的内容要恢复执行， 
```python
def node_a(state: State):
    # ✅ 正确用法：使用幂等的 upsert 操作
    # 多次运行结果一致
    db.upsert_user(
        user_id=state["user_id"],
        status="pending_approval"
    )

    approved = interrupt("Approve this change?")

    return {"approved": approved}
```
```
  db.upsert_user(
        user_id=state["user_id"],
        status="pending_approval"
    )
```
上面也要执行，多次修改是相同的，所以是幂等的，
   - 推荐写法1： 副作用放在断点之后
   - 副作用写在不同的节点

前面不呢个写a+=1，会重复执行， 也不能写.create, 也不能写append，会增加两次， 正确写法，应该放在后面

就是中断要遵循底层逻辑

# 中断在底层的检查点如何存储的
前面说中断在底层是以来checkpoint实现的，这里看下怎么存储的

In [1]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

class OverAllState(TypedDict):
    username: str
    age: int

def node_a(state: OverAllState) -> OverAllState:
    username = interrupt("请输入您的姓名")
    age = interrupt("请输入您的年龄")
    return {
        "username": username,
        "age": age
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "123"}}
username_interrupted_res = graph.invoke({}, config=config)
print('=' * 30, '-> username_interrupted_res <-', '=' * 30)
print(username_interrupted_res)

============================== -> username_interrupted_res <- ==============================
{'__interrupt__': [Interrupt(value='请输入您的姓名', id='ef0e13db062f468d0385a8aed7e464a7')]}


In [2]:
username_interrupted_his = list(graph.get_state_history(config=config))
print('=' * 30, '-> username_interrupted_his <-', '=' * 30)
print(username_interrupted_his)


============================== -> username_interrupted_his <- ==============================
[StateSnapshot(values={}, next=('node_a',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c80-6718-8000-8b4cc199dece'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-17T07:43:18.215231+00:00', parent_config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, tasks=(PregelTask(id='cf2600c0-1b14-d617-8ca2-2ac54b73e642', name='node_a', path=('__pregel_pull', 'node_a'), error=None, interrupts=(Interrupt(value='请输入您的姓名', id='ef0e13db062f468d0385a8aed7e464a7'),), state=None, result=None),), interrupts=(Interrupt(value='请输入您的姓名', id='ef0e13db062f468d0385a8aed7e464a7'),)), StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, metadata={'source': 

============================== -> username_interrupted_his <- ==============================
> 已经触发中断了，
[StateSnapshot(values={}, next=('node_a',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': **'1f1b26b7-2c80-6718-8000-8b4cc199dece'**}}, 
metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-17T07:43:18.215231+00:00', parent_config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, 
tasks=(PregelTask(id='cf2600c0-1b14-d617-8ca2-2ac54b73e642', name='node_a', path=('__pregel_pull', 'node_a'), error=None, interrupts=(Interrupt(value='请输入您的姓名', id='ef0e13db062f468d0385a8aed7e464a7'),), state=None, result=None),), 
interrupts=(Interrupt(value='请输入您的姓名', id=**'ef0e13db062f468d0385a8aed7e464a7'**),)), 
多了interrput

> 忽略
StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-09-17T07:43:18.211039+00:00', parent_config=None, tasks=(PregelTask(id='574db721-d84b-f667-af64-b386a14f748e', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={}),), interrupts=())]

In [3]:

age_interrupted_res = graph.invoke(Command(resume="小黄"), config=config)
print('=' * 30, '-> age_interrupted_res <-', '=' * 30)
print(age_interrupted_res)


============================== -> age_interrupted_res <- ==============================
{'__interrupt__': [Interrupt(value='请输入您的年龄', id='ef0e13db062f468d0385a8aed7e464a7')]}


In [ ]:
age_interrupted_his = list(graph.get_state_history(config=config)) # 获取最新的
print('=' * 30, '-> age_interrupted_his <-', '=' * 30)
print(age_interrupted_his)


============================== -> age_interrupted_his <- ==============================
[StateSnapshot(values={}, next=('node_a',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c80-6718-8000-8b4cc199dece'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-17T07:43:18.215231+00:00', parent_config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, tasks=(PregelTask(id='cf2600c0-1b14-d617-8ca2-2ac54b73e642', name='node_a', path=('__pregel_pull', 'node_a'), error=None, interrupts=(Interrupt(value='请输入您的年龄', id='ef0e13db062f468d0385a8aed7e464a7'),), state=None, result={}),), interrupts=(Interrupt(value='请输入您的年龄', id='ef0e13db062f468d0385a8aed7e464a7'),)), StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, metadata={'source': 'input'

============================== -> age_interrupted_his <- ==============================
[StateSnapshot(values={}, next=('node_a',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': **'1f1b26b7-2c80-6718-8000-8b4cc199dece'**更新覆盖}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-17T07:43:18.215231+00:00', parent_config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, tasks=(PregelTask(id='cf2600c0-1b14-d617-8ca2-2ac54b73e642', name='node_a', path=('__pregel_pull', 'node_a'), error=None, interrupts=(Interrupt(value='请输入您的年龄', id='ef0e13db062f468d0385a8aed7e464a7'),), state=None, result={}),), interrupts=(Interrupt(value='请输入您的年龄', id=**'ef0e13db062f468d0385a8aed7e464a7'**中断的id也一样更新覆盖，小黄哪些值也是不可见的),)),

> 第二次执行
StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-09-17T07:43:18.211039+00:00', parent_config=None, tasks=(PregelTask(id='574db721-d84b-f667-af64-b386a14f748e', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={}),), interrupts=())]

In [5]:

resumed_res = graph.invoke(Command(resume=123), config=config)
print('=' * 30, '-> resumed_res <-', '=' * 30)
print(resumed_res)


============================== -> resumed_res <- ==============================
{'username': '小黄', 'age': 123}


In [6]:
resumed_his = list(graph.get_state_history(config=config))
print('=' * 30, '-> resumed_his <-', '=' * 30)
print(resumed_his)

============================== -> resumed_his <- ==============================
[StateSnapshot(values={'username': '小黄', 'age': 123}, next=(), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26cd-8161-6154-8001-5ea68d2e3624'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-09-17T07:53:17.673282+00:00', parent_config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c80-6718-8000-8b4cc199dece'}}, tasks=(), interrupts=()), StateSnapshot(values={}, next=('node_a',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c80-6718-8000-8b4cc199dece'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-17T07:43:18.215231+00:00', parent_config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, tasks=(PregelTask(id='cf2600c0-1b14-d617-8ca2-2ac54b73e642', name='nod

> 执行完成后，打印所有的检查点
[StateSnapshot(values={'username': '小黄', 'age': 123}, next=(), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26cd-8161-6154-8001-5ea68d2e3624'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-09-17T07:53:17.673282+00:00', parent_config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c80-6718-8000-8b4cc199dece'}}, tasks=(), interrupts=()),



 StateSnapshot(values={}, next=('node_a',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': **'1f1b26b7-2c80-6718-8000-8b4cc199dece'还是一样**}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-17T07:43:18.215231+00:00', parent_config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, 
 
- tasks=(PregelTask(id='cf2600c0-1b14-d617-8ca2-2ac54b73e642', name='node_a', path=('__pregel_pull', 'node_a'), error=None, interrupts=(Interrupt(value='请输入您的年龄', id='ef0e13db062f468d0385a8aed7e464a7'),), state=None, result={'username': '小黄', 'age': 123}),), 
- interrupts=(Interrupt(value='请输入您的年龄', id=**'ef0e13db062f468d0385a8aed7e464a7'**也一样),)), 



StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '123', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26b7-2c76-6327-bfff-c6b3919a33d3'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-09-17T07:43:18.211039+00:00', parent_config=None, tasks=(PregelTask(id='574db721-d84b-f667-af64-b386a14f748e', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={}),), interrupts=())]

重点内容， tasks里和interrupts存储了一样的两份interrupt_id，因为我们只有一个节点，不涉及多个同时触发中断的情况，  所以看起来两种没有区别，实际上有区别

中断信息存了两份：

- **`StateSnapshot`** 的 **`tasks`** 属性下记录的 **`PregelTask`** 实例的 **`interrupts`** 字段记录了当前任务触发的中断信息。  # 当前任务触发的，谁导致弹出了

- **`StateSnapshot`** 的 **`interrupts`** 属性记录了当前超步发生的所有中断  # 整个是全局所有某个超步的中断

当存在多个并行中断时

- 中断信息被记录在各自所属任务的 **`PregelTask`** 实例中
- 同时也会被汇总在 **`StateSnapshot`** 的 **`tasks`** 属性中


---
**`checkpoint_id`、任务`id`、中断`id`** 和历史检查点完全一致，只是 **`Interrupt`** 实例的 **`value`** 属性变成了第二次中断的值。

由此可见，恢复运行时检查点的更新确实是覆盖写入，历史中断信息不会被保留。